OverviewThis notebook implements the Bronze to Silver layer of the Medallion Architecture. It performs data profiling, DQX validation, quarantine handling, cleaning transformations, and writes clean data to Silver tables.

Bronze Table → Data Profiling → DQX Validation → [Valid Records] → Cleaning → Silver Table → [Invalid Records] → Quarantine Table

Source and Target
**Current Testing Mode:** Reads from chinook_azure_sql_catalog.chinook.<TableName>
**Production Mode (change before integration):** Should read from chinook_catalog.bronze.<tablename>
**Target:** Writes to chinook_catalog.silver.<tablename>
**Quarantine:** Writes to chinook_catalog.silver.quarantine_<tablename>
**Execution Log:** Writes to chinook_catalog.silver.dqx_execution_log

Tables Processed (11 tables)
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track

DQX Validation Rules by Table
Album (6 rules)
album_id_not_null: AlbumId cannot be null
album_id_positive: AlbumId must be greater than 0
title_not_null: Title cannot be null
title_not_empty: Title must have at least 1 character
artist_id_not_null: ArtistId cannot be null
artist_id_positive: ArtistId must be greater than 0

Artist (4 rules)
artist_id_not_null: ArtistId cannot be null
artist_id_positive: ArtistId must be greater than 0
name_not_null: Name cannot be null
name_not_empty: Name must have at least 1 character

Customer (9 rules)
customer_id_not_null: CustomerId cannot be null
customer_id_positive: CustomerId must be greater than 0
first_name_not_null: FirstName cannot be null
first_name_not_empty: FirstName must have at least 1 character
last_name_not_null: LastName cannot be null
last_name_not_empty: LastName must have at least 1 character
email_not_null: Email cannot be null
email_valid_format: Email must match regex pattern for valid email
support_rep_id_positive: SupportRepId must be greater than or equal to 0

Employee (9 rules)
employee_id_not_null: EmployeeId cannot be null
employee_id_positive: EmployeeId must be greater than 0
first_name_not_null: FirstName cannot be null
first_name_not_empty: FirstName must have at least 1 character
last_name_not_null: LastName cannot be null
last_name_not_empty: LastName must have at least 1 character
email_valid_format: Email must match regex pattern for valid email
hire_date_not_null: HireDate cannot be null
reports_to_positive_if_exists: ReportsTo must be greater than 0 if present

Genre (4 rules)
genre_id_not_null: GenreId cannot be null
genre_id_positive: GenreId must be greater than 0
name_not_null: Name cannot be null
name_not_empty: Name must have at least 1 character

Invoice (10 rules)
invoice_id_not_null: InvoiceId cannot be null
invoice_id_positive: InvoiceId must be greater than 0
customer_id_not_null: CustomerId cannot be null
customer_id_positive: CustomerId must be greater than 0
invoice_date_not_null: InvoiceDate cannot be null
total_not_null: Total cannot be null
total_non_negative: Total must be greater than or equal to 0
billing_address_not_null: BillingAddress cannot be null
billing_city_not_null: BillingCity cannot be null
billing_country_not_null: BillingCountry cannot be null

InvoiceLine (12 rules)
invoice_line_id_not_null: InvoiceLineId cannot be null
invoice_line_id_positive: InvoiceLineId must be greater than 0
invoice_id_not_null: InvoiceId cannot be null
invoice_id_positive: InvoiceId must be greater than 0
track_id_not_null: TrackId cannot be null
track_id_positive: TrackId must be greater than 0
unit_price_not_null: UnitPrice cannot be null
unit_price_positive: UnitPrice must be greater than 0
unit_price_reasonable: UnitPrice must be less than 100
quantity_not_null: Quantity cannot be null
quantity_positive: Quantity must be greater than 0
quantity_reasonable: Quantity must be less than 1000

MediaType (4 rules)
media_type_id_not_null: MediaTypeId cannot be null
media_type_id_positive: MediaTypeId must be greater than 0
name_not_null: Name cannot be null
name_not_empty: Name must have at least 1 character

Playlist (4 rules)
playlist_id_not_null: PlaylistId cannot be null
playlist_id_positive: PlaylistId must be greater than 0
name_not_null: Name cannot be null
name_not_empty: Name must have at least 1 character

PlaylistTrack (4 rules)
playlist_id_not_null: PlaylistId cannot be null
playlist_id_positive: PlaylistId must be greater than 0
track_id_not_null: TrackId cannot be null
track_id_positive: TrackId must be greater than 0

Track (15 rules)
track_id_not_null: TrackId cannot be null
track_id_positive: TrackId must be greater than 0
name_not_null: Name cannot be null
name_not_empty: Name must have at least 1 character
album_id_positive: AlbumId must be greater than 0
media_type_id_not_null: MediaTypeId cannot be null
media_type_id_positive: MediaTypeId must be greater than 0
genre_id_positive: GenreId must be greater than 0
milliseconds_not_null: Milliseconds cannot be null
milliseconds_positive: Milliseconds must be greater than 0
milliseconds_reasonable: Milliseconds must be less than 3600000 (1 hour)
bytes_positive: Bytes must be greater than 0
unit_price_not_null: UnitPrice cannot be null
unit_price_positive: UnitPrice must be greater than 0
unit_price_reasonable: UnitPrice must be less than 50

Total: 81 validation rules across 11 tables

Data Profiling (performed for each table)

Total row count
Total column count
For each column: data type, null count, null percentage, distinct count
Duplicate row count


Cleaning Transformations
Applied to all tables:

Trim whitespace from all string columns
Replace empty strings with null
Add _loaded_at timestamp column

Customer-specific:

Create full_name column (FirstName + LastName)
Uppercase Country column

Employee-specific:

Create full_name column (FirstName + LastName)
Uppercase Country column

Invoice-specific:

Convert InvoiceDate to date type


Quarantine Strategy

Failed records are written to separate quarantine tables per source table
Quarantine table naming: quarantine_<tablename>
Additional columns added: dqx_failure_reason (explains which rule failed), dqx_failed_at (timestamp)
Records that pass validation go to Silver, records that fail go to quarantine


DQX Execution Logging
For every table processed, a log entry is written to chinook_catalog.silver.dqx_execution_log containing:

table_name
execution_time
total_records_processed
records_passed
records_failed
status (SUCCESS or FAILED)

Silver Tables Created

chinook_catalog.silver.album
chinook_catalog.silver.artist
chinook_catalog.silver.customer
chinook_catalog.silver.employee
chinook_catalog.silver.genre
chinook_catalog.silver.invoice
chinook_catalog.silver.invoiceline
chinook_catalog.silver.mediatype
chinook_catalog.silver.playlist
chinook_catalog.silver.playlisttrack
chinook_catalog.silver.track
chinook_catalog.silver.quarantine_customer (and other quarantine tables as needed)
chinook_catalog.silver.dqx_execution_log

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04_bronze_to_silver
# MAGIC This notebook performs:
# MAGIC 1. Data profiling on Bronze tables
# MAGIC 2. DQX validation with quality rules
# MAGIC 3. Routes failed records to quarantine tables with failure reasons
# MAGIC 4. Applies cleaning transformations
# MAGIC 5. Writes clean data to Silver tables

# MAGIC %md
# MAGIC ### Install DQX Library

In [0]:
%pip install databricks-labs-dqx

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

In [0]:
catalog_name = "chinook_catalog"
bronze_schema = "bronze"
silver_schema = "silver"

# Read active tables from metadata
active_tables_df = spark.sql(f"""
    SELECT table_name 
    FROM {catalog_name}.{meta_schema}.parent_metadata
    WHERE active_flag = 'Y'
""")

tables_to_process = [row["table_name"] for row in active_tables_df.collect()]

# List of all tables to process
#tables_to_process = [
#    "Album",
#    "Artist",
#    "Customer",
#    "Employee",
#    "Genre",
#    "Invoice",
#    "InvoiceLine",
#    "MediaType",
#    "Playlist",
#    "PlaylistTrack",
#    "Track"
#]

In [0]:
def get_bronze_table_path(table_name):
    return f"{catalog_name}.{bronze_schema}.{table_name.lower()}"

def get_silver_table_path(table_name):
    return f"{catalog_name}.{silver_schema}.{table_name.lower()}"

def get_quarantine_table_path(table_name):
    return f"{catalog_name}.{silver_schema}.quarantine_{table_name.lower()}"

In [0]:
def profile_table(df, table_name):
    """
    Profile a dataframe and return profiling statistics
    """
    print(f"\n{'='*60}")
    print(f"PROFILING: {table_name}")
    print(f"{'='*60}")
    
    total_rows = df.count()
    total_cols = len(df.columns)
    
    print(f"Total Rows: {total_rows}")
    print(f"Total Columns: {total_cols}")
    print(f"\nColumn Details:")
    print("-" * 60)
    
    profiling_results = []
    
    for col_name in df.columns:
        col_type = str(df.schema[col_name].dataType)
        null_count = df.filter(F.col(col_name).isNull()).count()
        null_pct = round((null_count / total_rows) * 100, 2) if total_rows > 0 else 0
        distinct_count = df.select(col_name).distinct().count()
        
        profiling_results.append({
            "column_name": col_name,
            "data_type": col_type,
            "null_count": null_count,
            "null_percentage": null_pct,
            "distinct_count": distinct_count
        })
        
        print(f"  {col_name}: {col_type} | Nulls: {null_count} ({null_pct}%) | Distinct: {distinct_count}")
    
    # Check for duplicates
    duplicate_count = total_rows - df.dropDuplicates().count()
    print(f"\nDuplicate Rows: {duplicate_count}")
    
    return profiling_results, total_rows, duplicate_count

In [0]:
def get_validation_rules(table_name):
    """
    Returns DQX validation rules for each table
    Using correct DQX function names and argument names
    """
    rules = {
        "Album": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "AlbumId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Title"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "ArtistId"}}}
        ],
        "Artist": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "ArtistId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Name"}}}
        ],
        "Customer": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "CustomerId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "FirstName"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "LastName"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Email"}}},
            {"name": "email_valid_format", "criticality": "error", "check": {"function": "regex_match", "arguments": {"column": "Email", "regex": "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"}}}
        ],
        "Employee": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "EmployeeId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "FirstName"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "LastName"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "HireDate"}}},
            {"name": "email_valid_format", "criticality": "error", "check": {"function": "regex_match", "arguments": {"column": "Email", "regex": "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"}}}
        ],
        "Genre": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "GenreId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Name"}}}
        ],
        "Invoice": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "InvoiceId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "CustomerId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "InvoiceDate"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Total"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "BillingAddress"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "BillingCity"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "BillingCountry"}}}
        ],
        "InvoiceLine": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "InvoiceLineId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "InvoiceId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "TrackId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "UnitPrice"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Quantity"}}},
            {"criticality": "error", "check": {"function": "sql_expression", "arguments": {"expression": "UnitPrice > 0", "msg": "UnitPrice must be positive"}}},
            {"criticality": "error", "check": {"function": "sql_expression", "arguments": {"expression": "Quantity > 0", "msg": "Quantity must be positive"}}}
        ],
        "MediaType": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "MediaTypeId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Name"}}}
        ],
        "Playlist": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "PlaylistId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Name"}}}
        ],
        "PlaylistTrack": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "PlaylistId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "TrackId"}}}
        ],
        "Track": [
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "TrackId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Name"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "MediaTypeId"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Milliseconds"}}},
            {"criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "UnitPrice"}}},
            {"criticality": "error", "check": {"function": "sql_expression", "arguments": {"expression": "Milliseconds > 0", "msg": "Milliseconds must be positive"}}},
            {"criticality": "error", "check": {"function": "sql_expression", "arguments": {"expression": "UnitPrice > 0", "msg": "UnitPrice must be positive"}}}
        ]
    }
    
    return rules.get(table_name, [])

In [0]:
def validate_with_dqx(df, table_name):
    """
    Validate dataframe using DQX rules
    Returns: valid_df, invalid_df, validation_summary
    """
    print(f"\n{'='*60}")
    print(f"DQX VALIDATION: {table_name}")
    print(f"{'='*60}")
    
    rules = get_validation_rules(table_name)
    
    if not rules:
        print(f"No validation rules defined for {table_name}. Passing all records.")
        return df, None, {"total": df.count(), "valid": df.count(), "invalid": 0}
    
    dq_engine = DQEngine(WorkspaceClient())
    
    # Use apply_checks_by_metadata for dict-based rules
    validated_df = dq_engine.apply_checks_by_metadata(df, rules)
    
    # Check if _errors column exists
    if "_errors" not in validated_df.columns:
        print(f"No _errors column found. Passing all records.")
        return df, None, {"total": df.count(), "valid": df.count(), "invalid": 0}
    
    # Filter valid records - where _errors is null or empty
    valid_df = validated_df.filter(
        F.col("_errors").isNull() | (F.size(F.col("_errors")) == 0)
    ).drop("_errors", "_warnings")
    
    # Filter invalid records - where _errors has entries
    invalid_df = validated_df.filter(
        F.col("_errors").isNotNull() & (F.size(F.col("_errors")) > 0)
    )
    
    total_count = df.count()
    valid_count = valid_df.count()
    invalid_count = invalid_df.count()
    
    print(f"Total Records: {total_count}")
    print(f"Valid Records: {valid_count}")
    print(f"Invalid Records: {invalid_count}")
    
    validation_summary = {
        "total": total_count,
        "valid": valid_count,
        "invalid": invalid_count
    }
    
    return valid_df, invalid_df, validation_summary

In [0]:
def apply_cleaning_transformations(df, table_name):
    """
    Apply cleaning transformations to the dataframe
    - Trim string columns
    - Standardize nulls
    - Additional table-specific cleaning
    """
    print(f"\nApplying cleaning transformations for {table_name}...")
    
    # Get string columns
    string_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]
    
    # Trim all string columns
    for col_name in string_columns:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))
    
    # Replace empty strings with null for string columns
    for col_name in string_columns:
        df = df.withColumn(col_name, F.when(F.col(col_name) == "", None).otherwise(F.col(col_name)))
    
    # Table-specific transformations
    if table_name == "Customer":
        # Create full_name column
        df = df.withColumn("full_name", F.concat_ws(" ", F.col("FirstName"), F.col("LastName")))
        # Standardize country names (uppercase)
        df = df.withColumn("Country", F.upper(F.col("Country")))
    
    if table_name == "Employee":
        # Create full_name column
        df = df.withColumn("full_name", F.concat_ws(" ", F.col("FirstName"), F.col("LastName")))
        # Standardize country names (uppercase)
        df = df.withColumn("Country", F.upper(F.col("Country")))
    
    if table_name == "Invoice":
        # Ensure InvoiceDate is date type
        df = df.withColumn("InvoiceDate", F.to_date(F.col("InvoiceDate")))
    
    # Add metadata columns
    df = df.withColumn("_loaded_at", F.current_timestamp())
    
    print(f"Cleaning complete for {table_name}")
    
    return df


In [0]:
def write_to_quarantine(invalid_df, table_name):
    """
    Write invalid records to quarantine table with failure reasons
    """
    if invalid_df is None or invalid_df.count() == 0:
        print(f"No invalid records to quarantine for {table_name}")
        return
    
    quarantine_path = get_quarantine_table_path(table_name)
    
    # Extract message field from each struct in _errors array
    quarantine_df = invalid_df.withColumn(
        "dqx_failure_reason",
        F.concat_ws("; ", F.transform(F.col("_errors"), lambda x: x.getField("message")))
    ).withColumn(
        "dqx_failed_at",
        F.current_timestamp()
    ).drop("_errors", "_warnings")
    
    quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_path)
    
    print(f"Written {quarantine_df.count()} invalid records to {quarantine_path}")

In [0]:
def write_to_silver(df, table_name):
    """
    Write clean records to Silver table
    """
    silver_path = get_silver_table_path(table_name)
    
    # Write to Silver table (overwrite mode as per Bronze behavior)
    df.write.format("delta").mode("overwrite").saveAsTable(silver_path)
    
    print(f"Written {df.count()} records to {silver_path}")

In [0]:
def log_dqx_execution(table_name, profiling_results, validation_summary, status):
    """
    Log DQX execution metrics
    """
    log_entry = {
        "table_name": table_name,
        "execution_time": datetime.now().isoformat(),
        "total_records_processed": validation_summary["total"],
        "records_passed": validation_summary["valid"],
        "records_failed": validation_summary["invalid"],
        "status": status
    }
    
    log_df = spark.createDataFrame([log_entry])
    
    log_table_path = f"{catalog_name}.{silver_schema}.dqx_execution_log"
    
    log_df.write.format("delta").mode("append").saveAsTable(log_table_path)
    
    print(f"\nDQX execution logged for {table_name}")
    return log_entry

In [0]:
def process_table(table_name):
    """
    Main function to process a single table through Bronze to Silver
    """
    print(f"\n{'#'*70}")
    print(f"PROCESSING TABLE: {table_name}")
    print(f"{'#'*70}")
    
    try:
        # Read directly from Azure SQL for testing
        source_path = f"chinook_catalog.bronze.{table_name}"
        print(f"\nReading from: {source_path}")
        df = spark.read.table(source_path)
        
        profiling_results, total_rows, duplicate_count = profile_table(df, table_name)
        
        valid_df, invalid_df, validation_summary = validate_with_dqx(df, table_name)
        
        write_to_quarantine(invalid_df, table_name)
        
        cleaned_df = apply_cleaning_transformations(valid_df, table_name)
        
        write_to_silver(cleaned_df, table_name)
        
        log_dqx_execution(table_name, profiling_results, validation_summary, "SUCCESS")
        
        print(f"\n✓ {table_name} processed successfully")
        return True
        
    except Exception as e:
        print(f"\n✗ Error processing {table_name}: {str(e)}")
        log_dqx_execution(table_name, [], {"total": 0, "valid": 0, "invalid": 0}, "FAILED")
        return False

In [0]:
results = {}
for table_name in tables_to_process:
    success = process_table(table_name)
    results[table_name] = "SUCCESS" if success else "FAILED"


######################################################################
PROCESSING TABLE: Album
######################################################################

Reading from: chinook_catalog.bronze.Album

PROFILING: Album
Total Rows: 347
Total Columns: 3

Column Details:
------------------------------------------------------------
  AlbumId: IntegerType() | Nulls: 0 (0.0%) | Distinct: 347
  Title: StringType() | Nulls: 0 (0.0%) | Distinct: 347
  ArtistId: IntegerType() | Nulls: 0 (0.0%) | Distinct: 204

Duplicate Rows: 0

DQX VALIDATION: Album
Total Records: 347
Valid Records: 347
Invalid Records: 0
No invalid records to quarantine for Album

Applying cleaning transformations for Album...
Cleaning complete for Album
Written 347 records to chinook_catalog.silver.album

DQX execution logged for Album

✓ Album processed successfully

######################################################################
PROCESSING TABLE: Artist
######################################################

In [0]:
print("\n" + "="*70)
print("BRONZE TO SILVER PIPELINE SUMMARY")
print("="*70)
for table_name, status in results.items():
    status_icon = "✓" if status == "SUCCESS" else "✗"
    print(f"{status_icon} {table_name}: {status}")
print("="*70)


BRONZE TO SILVER PIPELINE SUMMARY
✓ Album: SUCCESS
✓ Artist: SUCCESS
✓ Customer: SUCCESS
✓ Employee: SUCCESS
✓ Genre: SUCCESS
✓ Invoice: SUCCESS
✓ InvoiceLine: SUCCESS
✓ MediaType: SUCCESS
✓ Playlist: SUCCESS
✓ PlaylistTrack: SUCCESS
✓ Track: SUCCESS


In [0]:
# Check all quarantine tables
quarantine_tables = [
    "quarantine_album",
    "quarantine_artist",
    "quarantine_customer",
    "quarantine_employee",
    "quarantine_genre",
    "quarantine_invoice",
    "quarantine_invoiceline",
    "quarantine_mediatype",
    "quarantine_playlist",
    "quarantine_playlisttrack",
    "quarantine_track"
]

for table in quarantine_tables:
    try:
        df = spark.read.table(f"chinook_catalog.silver.{table}")
        count = df.count()
        if count > 0:
            print(f"\n{'='*60}")
            print(f"{table}: {count} quarantined records")
            print(f"{'='*60}")
            df.display()
    except:
        print(f"{table}: No quarantine table exists (no failures)")

quarantine_album: No quarantine table exists (no failures)
quarantine_artist: No quarantine table exists (no failures)

quarantine_customer: 11 quarantined records


CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId,dqx_failure_reason,dqx_failed_at
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T20:22:22.330Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T00:11:49.764Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T16:59:56.199Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T01:24:57.525Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T18:08:46.382Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T17:08:26.605Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T00:57:58.897Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T17:17:58.298Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T15:18:14.451Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-04-02T03:52:00.794Z


quarantine_employee: No quarantine table exists (no failures)
quarantine_genre: No quarantine table exists (no failures)
quarantine_invoice: No quarantine table exists (no failures)
quarantine_invoiceline: No quarantine table exists (no failures)
quarantine_mediatype: No quarantine table exists (no failures)
quarantine_playlist: No quarantine table exists (no failures)
quarantine_playlisttrack: No quarantine table exists (no failures)
quarantine_track: No quarantine table exists (no failures)


In [0]:
spark.read.table("chinook_catalog.silver.quarantine_customer").display()

CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId,dqx_failure_reason,dqx_failed_at
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T20:22:22.330Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T00:11:49.764Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T16:59:56.199Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T01:24:57.525Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T18:08:46.382Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T17:08:26.605Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-29T00:57:58.897Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T17:17:58.298Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-03-28T15:18:14.451Z
49,Stanisław,Wójcik,null,Ordynacka 10,Warsaw,null,Poland,00-358,+48 22 828 37 39,null,stanisław.wójcik@wp.pl,4,Column 'Email' is not matching regex,2026-04-02T03:52:00.794Z
